# T2.1 - Re-evaluate v19 multi-scale (L0 + L1) at 9,128-slide scale

**Purpose:** Close thesis review item T2.1. Section 4.8 of the thesis rejects the v19 multi-scale variant on the basis of a 600-slide development-scale experiment (`MS_TopK` reached QWK = 0.6262 +/- 0.0798 vs. v19 single-scale baseline 0.7821 +/- 0.0338). Section 4.6 argues that 600-slide-scale comparisons are systematically underpowered. The two claims are internally inconsistent: the rejection-as-formulation-specific argument relies on a scale that the power-scaling lesson labels insufficient.

This notebook re-runs the v19 multi-scale variant at the same 9,128-slide PANDA noise-cleaned cohort used for the v20 factorial, so the rejection can be empirically supported (or overturned) at the same scale as the rest of Chapter 3.

**Source code anchors:**
- `CLAM-master/final-code/03-optimization/CLAM_TR_v19_MultiScale_Corrected.ipynb` cells 1.6 (model), 0.8b (KDTree matching), 0.9 (datasets), 1.1 (DecayAttentionBlock), 1.8 (losses), 2.1-2.3 (training/5-fold CV).
- `CLAM-master/final-code/04-final-v20/CLAM_TR_v20_FullScale_Colab.ipynb` cell 6 (9,128 noise-cleaned CSV path), cell 37 (kfold 1-5 -> 0-4 conversion), cell 19 (`PANDADataset` RAM-cache loader, used here for L0 features only).
- `CLAM-master/CLAM_TR_Results/v19_multiscale/v19_final_summary.json` (the 600-slide MS_TopK number quoted in Section 4.8).

**Runtime warning:** This is a 12-24 hour Colab Pro+ run on an A100. T4 will not be sufficient (insufficient GPU RAM for L1 + L0 features simultaneously plus the larger v19 multi-scale model). 5 folds x ~50 epochs x ~9,128 slides x both-scales-per-slide IO ~ 2.5-4 hr/fold on A100. The notebook writes a per-fold checkpoint and per-fold predictions to `OUTPUT_DIR` so partial progress survives Colab disconnects.

**Prerequisites status as of S109e.5 verification (all MET — just press run):**
- L0 (20x) UNI features at `/content/drive/MyDrive/UNI_features/`: 9,128 `.pt` files (100% cohort). ✓
- L1 (5x cytology) UNI features at `/content/drive/MyDrive/CLAM_TR_Features/L1_P2_provider/uni/`: 8,771 `.pt` files (95.15% in-cohort; Phase 1 extraction complete). ✓
- PANDA noise-cleaned CSV (`train-5kfold_remove_noisy_by_0622_rad_13_08_ka_15_10.csv`): exists on Drive. ✓
- Cell 3 hard-gate prerequisites check will report PASS for all three.

**Just press run (Colab Pro+ on A100 recommended):**
1. Mount Drive (cell 3 does this automatically).
2. Wait for prereq check to PASS (~30 sec).
3. Run cells 5 through 13 — that's the full 5-fold sweep (`TEST_MODE = False` since S109e.8). Cell 11's `fold_runner` is **resume-safe**: if Colab disconnects mid-fold, re-run cell 13 and it picks up from the last saved checkpoint.
4. Cell 15 produces the paired comparison vs §4.8 600-slide baseline. Cell 17 prints the defence-ready summary.

**Outputs on Drive (`CLAM_TR_Results/t21_v19_multiscale_9128/`):**
- `checkpoints/v19_original_fold{0..4}_best.pt` — per-fold best-epoch model state + metrics.
- `predictions/v19_original_fold{0..4}_predictions.json` — per-fold y_true / y_pred.
- `matching_cache/` — L0-L1 KDTree match cache (speeds up re-runs).
- `v19_original_summary.json` — combined 5-fold metrics + QWK mean ± std.


## Cell 2 markdown -- Prerequisites check

Cell 3 below verifies, BEFORE any training cell runs, that:
- The 9,128-slide noise-cleaned CSV is present.
- L0 features (20x architecture; UNI, 1024d, `.pt`) are present for >= 95% of CSV slides.
- L1 features (cytology scale; UNI, 1024d, `.pt`) are present for >= 95% of CSV slides.

If L1 coverage is below 95% (the expected state as of writing), Cell 3 raises a clear error pointing at the L1 feature-extraction notebook Mohamed would need to run first. **No subsequent cell will execute** because the dataset loader expects both scales.


In [ ]:
# T2.1 Cell 3 - Prerequisites check. HARD GATE.
# Mount Drive + verify CSV + verify both feature scales cover the cohort.
from google.colab import drive
drive.mount('/content/drive')

import os, glob, sys
import pandas as pd

# Paths (edit these to match your Drive layout; defaults match the v20 fullscale notebook).
DRIVE_ROOT = '/content/drive/MyDrive'
LABELS_PATH = os.path.join(
    DRIVE_ROOT,
    'panda-data/prostate-cancer-grade-assessment',
    'Kaggle-PANDA-1st-place-solution-master/input/'
    'train-5kfold_remove_noisy_by_0622_rad_13_08_ka_15_10.csv')

# L0 = 20x architecture scale (UNI), shared with v20 fullscale.
LEVEL0_FEATURES_DIR = os.path.join(DRIVE_ROOT, 'UNI_features')
# L1 = 5x cytology scale (UNI) per v19 CONFIG['features_l1_dir']. P2_provider was the v19 recipe.
# At full-cohort scale you may use any L1_P*_provider recipe consistent with the L0 patch grid;
# Mohamed should pick the same recipe he ran for the 600-slide v19 §4.8 experiment.
LEVEL1_FEATURES_DIR = os.path.join(DRIVE_ROOT, 'CLAM_TR_Features/L1_P2_provider/uni')

L0_EXT = '.pt'
L1_EXT = '.pt'

# Coverage threshold below which we refuse to proceed (avoids accidental 600-slide re-run).
MIN_COVERAGE_FRAC = 0.95
# Below this absolute slide count we know L1 hasn't been extracted at full scale.
FULL_COHORT_SIZE = 9128

print('=' * 70)
print('T2.1 PREREQUISITES CHECK')
print('=' * 70)

# (i) CSV present.
if not os.path.exists(LABELS_PATH):
    raise FileNotFoundError(
        f'9,128-slide noise-cleaned CSV not found at:\n  {LABELS_PATH}\n'
        'This file is shipped with the Kaggle-PANDA-1st-place-solution repository\n'
        '(input/train-5kfold_remove_noisy_by_0622_rad_13_08_ka_15_10.csv).\n'
        'Copy it to your Drive at the path above before re-running.')
df_csv = pd.read_csv(LABELS_PATH)
print(f'[CSV] {len(df_csv):,} rows. Columns: {list(df_csv.columns)}')
assert 'image_id' in df_csv.columns and 'isup_grade' in df_csv.columns and 'kfold' in df_csv.columns, \
    'CSV must contain image_id, isup_grade, kfold columns.'
all_ids = set(df_csv['image_id'].astype(str).tolist())
print(f'[CSV] unique slide IDs: {len(all_ids):,}')

# (ii) L0 coverage.
if not os.path.isdir(LEVEL0_FEATURES_DIR):
    raise FileNotFoundError(
        f'L0 features directory missing:\n  {LEVEL0_FEATURES_DIR}\n'
        'This is the 20x UNI feature directory shared with the v20 fullscale run.\n'
        'It contains ~9,128 .pt files keyed by slide ID. Re-extract via\n'
        'CLAM-master/final-code/01-data-preparation/UNI_Feature_Extraction.ipynb if missing.')
l0_ids = {os.path.splitext(os.path.basename(p))[0]
          for p in glob.glob(os.path.join(LEVEL0_FEATURES_DIR, f'*{L0_EXT}'))}
l0_covered = all_ids & l0_ids
l0_frac = len(l0_covered) / max(1, len(all_ids))
print(f'[L0] {len(l0_ids):,} feature files, {len(l0_covered):,}/{len(all_ids):,} '
      f'CSV slides covered ({l0_frac:.1%}).')
if l0_frac < MIN_COVERAGE_FRAC:
    raise RuntimeError(
        f'L0 coverage {l0_frac:.1%} is below threshold {MIN_COVERAGE_FRAC:.0%}. '
        'Re-run UNI L0 feature extraction before proceeding.')

# (iii) L1 coverage. THIS IS THE EXPECTED FAILURE MODE.
if not os.path.isdir(LEVEL1_FEATURES_DIR):
    raise FileNotFoundError(
        f'L1 features directory missing:\n  {LEVEL1_FEATURES_DIR}\n\n'
        'L1 (cytology-scale, level-1) UNI features are REQUIRED for v19 multi-scale.\n'
        'As of the thesis snapshot the only L1 features that exist on Drive are\n'
        'the 600-slide development subset under CLAM_TR_Features/L1_P*/uni/.\n\n'
        'You must extract L1 features for all 9,128 noise-cleaned slides FIRST.\n'
        'Suggested pipeline:\n'
        '  1. CLAM-master/clam-framework/create_patches_fp.py  with patch_level=1\n'
        '     (corresponds to the L1_P2_provider preset; 256x256 patches at level 1).\n'
        '  2. CLAM-master/CLAM_TR_Preprocessing/coordinates_level1/  for the coord cache.\n'
        '  3. CLAM-master/final-code/01-data-preparation/UNI_Feature_Extraction.ipynb\n'
        '     -- duplicate the notebook, redirect input/output to the level-1 patch and\n'
        '     L1 feature directories, re-run for the full 9,128 cohort.\n'
        '  Estimated wall time: 8-16 hours on A100, dominated by patch IO.')
l1_ids = {os.path.splitext(os.path.basename(p))[0]
          for p in glob.glob(os.path.join(LEVEL1_FEATURES_DIR, f'*{L1_EXT}'))}
l1_covered = all_ids & l1_ids
l1_frac = len(l1_covered) / max(1, len(all_ids))
print(f'[L1] {len(l1_ids):,} feature files, {len(l1_covered):,}/{len(all_ids):,} '
      f'CSV slides covered ({l1_frac:.1%}).')
if l1_frac < MIN_COVERAGE_FRAC:
    raise RuntimeError(
        f'L1 coverage {l1_frac:.1%} is below threshold {MIN_COVERAGE_FRAC:.0%}.\n'
        f'You currently have L1 features for {len(l1_ids):,} slides; the cohort needs '
        f'>= {int(FULL_COHORT_SIZE * MIN_COVERAGE_FRAC):,}.\n'
        'This matches the expected state at the time the T2.1 notebook was written:\n'
        'L1 was only extracted for the 600-slide development subset (P0-P6).\n\n'
        'Run the L1 feature-extraction pipeline described in the FileNotFoundError\n'
        'message above for the full 9,128 cohort, then re-run THIS cell.')

# (iv) Both scales cover the same slides? Restrict to the intersection.
both_ids = l0_covered & l1_covered
both_frac = len(both_ids) / max(1, len(all_ids))
print(f'[BOTH] L0 INTERSECTION L1: {len(both_ids):,}/{len(all_ids):,} slides ({both_frac:.1%}).')
if both_frac < MIN_COVERAGE_FRAC:
    raise RuntimeError(
        f'L0 INTERSECTION L1 coverage {both_frac:.1%} below threshold {MIN_COVERAGE_FRAC:.0%}. '
        'Some slides have L0 but not L1 or vice versa. Re-run the missing scale.')

print('=' * 70)
print('PREREQUISITES OK -- proceeding to setup.')
print('=' * 70)


## Setup: paths + Drive mount

Cell 5 below pins all paths to absolute Drive locations and creates the output directory. Edit `DRIVE_ROOT`, `LEVEL0_FEATURES_DIR`, `LEVEL1_FEATURES_DIR`, `OUTPUT_DIR` if your layout differs from the v20 fullscale conventions.


In [ ]:
# T2.1 Cell 5 - Paths + dependencies + reproducibility.
import subprocess, sys
for pkg in ['scipy', 'tqdm']:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', pkg], check=False)

import os, json, time, random, math
from collections import Counter
from typing import Dict, List, Optional, Tuple, Any

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim.lr_scheduler import CosineAnnealingLR, LambdaLR
from scipy.spatial import cKDTree
from sklearn.metrics import cohen_kappa_score, accuracy_score, balanced_accuracy_score
from scipy import stats
from tqdm.auto import tqdm

# Re-bind path constants (already set in Cell 3, but persist across runs).
DRIVE_ROOT = '/content/drive/MyDrive'
SPLIT_CSV_PATH = os.path.join(
    DRIVE_ROOT,
    'panda-data/prostate-cancer-grade-assessment',
    'Kaggle-PANDA-1st-place-solution-master/input/'
    'train-5kfold_remove_noisy_by_0622_rad_13_08_ka_15_10.csv')
LEVEL0_FEATURES_DIR = os.path.join(DRIVE_ROOT, 'UNI_features')
LEVEL1_FEATURES_DIR = os.path.join(DRIVE_ROOT, 'CLAM_TR_Features/L1_P2_provider/uni')
L0_EXT = '.pt'
L1_EXT = '.pt'

# Output directory holds per-fold checkpoints + per-fold predictions + matching cache.
OUTPUT_DIR = os.path.join(DRIVE_ROOT, 'CLAM_TR_Results', 't21_v19_multiscale_9128')
CHECKPOINT_DIR = os.path.join(OUTPUT_DIR, 'checkpoints')
PREDICTIONS_DIR = os.path.join(OUTPUT_DIR, 'predictions')
MATCHING_CACHE_DIR = os.path.join(OUTPUT_DIR, 'matching_cache')
for d in (OUTPUT_DIR, CHECKPOINT_DIR, PREDICTIONS_DIR, MATCHING_CACHE_DIR):
    os.makedirs(d, exist_ok=True)

# Reproducibility (matches v19 CONFIG['seed']).
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.backends.cudnn.deterministic = True
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'torch={torch.__version__}, device={device}')
if device.type == 'cuda':
    p = torch.cuda.get_device_properties(0)
    print(f'GPU: {p.name}, {p.total_memory/1e9:.1f} GB')
    if p.total_memory / 1e9 < 30:
        print('WARNING: GPU has <30 GB RAM. T4 may OOM on v19 multi-scale at 9,128 scale.')
        print('         Recommended: A100 40GB or A100 80GB.')

# Test/full mode.
TEST_MODE = False  # Full 5-fold sweep at 9,128-slide scale.
# Estimated runtime: ~12-24 hr on A100 40GB (5 folds x ~2-5 hr/fold).
# WARNING: T4 (15 GB) likely OOMs on v19 multi-scale with hidden_dim=512 + 4 heads.
# Recommended: cloud A100 (Lambda Cloud / RunPod / vast.ai; ~$30-50 budget).
# Resume-safe: cell 13's fold_runner skips folds whose checkpoint + predictions
# JSON both already exist, so Colab disconnects are recoverable by re-running.
# Set TEST_MODE = True to do a quick single-fold smoke test before the full sweep.
# Hyperparameter profile. 'v19_original' replicates the 600-slide §4.8 config verbatim.
# 'v20_bestHP' borrows the v20 best-HP defaults that worked at 9,128 scale.
HP_PROFILE = 'v19_original'    # one of {'v19_original', 'v20_bestHP'}

print(f'OUTPUT_DIR    = {OUTPUT_DIR}')
print(f'TEST_MODE     = {TEST_MODE}')
print(f'HP_PROFILE    = {HP_PROFILE}')


## v19 model class (extracted from CLAM_TR_v19_MultiScale_Corrected.ipynb)

Cell 7 below contains the verbatim multi-scale model (`R2FixedTopKMultiScale`) plus its `DecayAttentionBlock` dependency, both copied from `CLAM_TR_v19_MultiScale_Corrected.ipynb` cells 1.1 and 1.6.

**Architecture summary (for reviewer reference):**
1. Two independent linear encoders project L1 (1024d) and L0 (1024d) features to a 512d hidden space each.
2. A 2-layer tile scorer uses the L1 hidden representation ONLY to score each L1 patch.
3. TopK selection (`top_k=128` matches the §4.8 BEST_K) gathers the same TopK indices from BOTH scales plus the L1 coordinate grid.
4. The L1 and L0 hidden representations of the top-K patches are concatenated (1024d) and passed through a 2-layer fusion MLP.
5. A `DecayAttentionBlock` (8 heads in `HP_PROFILE='v20_bestHP'`, 4 heads in `'v19_original'`; learnable gamma; spatial distance decay in logit space) runs self-attention over the TopK patches.
6. Mean-pool over the TopK + linear CE classifier.


In [ ]:
# T2.1 Cell 7 - DecayAttentionBlock + R2FixedTopKMultiScale.
# Verbatim from CLAM_TR_v19_MultiScale_Corrected.ipynb cells 1.1 and 1.6.

from typing import Union

class DecayAttentionBlock(nn.Module):
    """Attention block with spatial distance-based decay (v19 cell 1.1).

    Args:
        embed_dim: Embedding dimension.
        num_heads: Number of attention heads.
        init_gamma: Initial gamma value(s). float OR list of per-head gammas.
        dropout: Dropout rate.
        learnable_gamma: True -> v19 (learnable, nn.Parameter in logit space).
                         False -> v18 (fixed, register_buffer).
    Decay is applied in LOGIT space as an additive bias `distance * log(gamma)`.
    """

    def __init__(self, embed_dim: int = 512, num_heads: int = 4,
                 init_gamma: Union[float, List[float]] = 0.9,
                 dropout: float = 0.1, learnable_gamma: bool = True):
        super().__init__()
        assert embed_dim % num_heads == 0, 'embed_dim must be divisible by num_heads'
        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads
        self.scale = self.head_dim ** -0.5
        self.learnable_gamma = learnable_gamma

        if isinstance(init_gamma, (list, tuple)):
            gamma_values = list(init_gamma)
            if len(gamma_values) < num_heads:
                gamma_values = gamma_values + [0.95] * (num_heads - len(gamma_values))
            gamma_values = gamma_values[:num_heads]
        else:
            gamma_values = [init_gamma] * num_heads
        for g in gamma_values:
            assert 0 < g < 1, f'gamma must be in (0,1), got {g}'

        if learnable_gamma:
            init_logits = [float(np.log(g / (1 - g))) for g in gamma_values]
            self.gamma_logit = nn.Parameter(torch.tensor(init_logits, dtype=torch.float32))
            self.gamma_fixed = None
        else:
            self.gamma_logit = None
            self.register_buffer('gamma_fixed',
                                 torch.tensor(gamma_values, dtype=torch.float32))

        self.W_qkv = nn.Linear(embed_dim, 3 * embed_dim)
        self.W_out = nn.Linear(embed_dim, embed_dim)
        self.norm = nn.LayerNorm(embed_dim)
        self.dropout = nn.Dropout(dropout)

    def _get_gamma(self) -> torch.Tensor:
        if self.learnable_gamma:
            return torch.sigmoid(self.gamma_logit)
        return self.gamma_fixed

    def forward(self, x: torch.Tensor, coords: torch.Tensor = None) -> torch.Tensor:
        B, N, C = x.shape
        H = self.num_heads
        D = self.head_dim
        qkv = self.W_qkv(x).reshape(B, N, 3, H, D).permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]                                          # [B,H,N,D]
        logits = (q @ k.transpose(-2, -1)) * self.scale                           # [B,H,N,N]

        if coords is not None:
            coords_2d = coords.squeeze(0) if coords.dim() == 3 else coords
            dx = coords_2d[:, 0:1] - coords_2d[:, 0:1].T
            dy = coords_2d[:, 1:2] - coords_2d[:, 1:2].T
            dist = torch.sqrt(dx ** 2 + dy ** 2 + 1e-8)
            dist = dist / (dist.max() + 1e-8)
        else:
            idx = torch.arange(N, device=x.device).float()
            dist = torch.abs(idx.unsqueeze(0) - idx.unsqueeze(1)) / (N + 1e-8)

        gamma = self._get_gamma().to(x.dtype)                                     # [H]
        log_gamma = torch.log(gamma + 1e-8)
        decay_bias = dist.unsqueeze(0) * log_gamma.view(H, 1, 1)                  # [H,N,N]
        decay_bias = decay_bias.unsqueeze(0).expand(B, -1, -1, -1)                # [B,H,N,N]
        logits = logits + decay_bias

        attn = F.softmax(logits, dim=-1)
        attn = self.dropout(attn)
        out = (attn @ v).transpose(1, 2).reshape(B, N, C)
        out = self.W_out(out)
        x = self.norm(x + self.dropout(out))
        return x


class R2FixedTopKMultiScale(nn.Module):
    """Multi-scale TopK model (PRIMARY, v19 cell 1.6).

    - Encodes L1 and L0 separately.
    - Scores tiles using L1 ONLY.
    - TopK gathers from BOTH scales.
    - Fuses per tile (concat + linear).
    - DecayAttentionBlock over the K fused tokens.
    - CE head only (CORAL variant exists in v19 cell 1.7; not used at §4.8).
    """

    def __init__(self, input_dim_l1: int = 1024, input_dim_l0: int = 1024,
                 hidden_dim: int = 512, n_classes: int = 5,
                 num_heads: int = 4, gamma: Union[float, List[float]] = 0.9,
                 dropout: float = 0.1, top_k: int = 256,
                 learnable_gamma: bool = True):
        super().__init__()
        self.n_classes = n_classes
        self.top_k = top_k
        self.hidden_dim = hidden_dim

        self.encoder_l1 = nn.Sequential(
            nn.Linear(input_dim_l1, hidden_dim), nn.ReLU(), nn.Dropout(dropout))
        self.encoder_l0 = nn.Sequential(
            nn.Linear(input_dim_l0, hidden_dim), nn.ReLU(), nn.Dropout(dropout))

        self.tile_scorer = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2), nn.Tanh(),
            nn.Linear(hidden_dim // 2, 1))

        self.fusion = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim * 2), nn.ReLU(), nn.Dropout(dropout))

        # NOTE: hidden_dim*2 must be divisible by num_heads.
        self.decay_attention = DecayAttentionBlock(
            hidden_dim * 2, num_heads, gamma, dropout, learnable_gamma)

        self.classifier = nn.Linear(hidden_dim * 2, n_classes)

    def forward(self, x_l1: torch.Tensor, x_l0: torch.Tensor,
                coords: torch.Tensor) -> torch.Tensor:
        B, N, _ = x_l1.shape
        h_l1 = self.encoder_l1(x_l1)
        h_l0 = self.encoder_l0(x_l0)
        scores = self.tile_scorer(h_l1).squeeze(-1)
        k = min(self.top_k, N)
        _, topk_idx = torch.topk(scores, k, dim=1)
        gather_idx_h = topk_idx.unsqueeze(-1).expand(-1, -1, self.hidden_dim)
        gather_idx_c = topk_idx.unsqueeze(-1).expand(-1, -1, 2)
        h_l1_topk = torch.gather(h_l1, 1, gather_idx_h)
        h_l0_topk = torch.gather(h_l0, 1, gather_idx_h)
        coords_topk = torch.gather(coords, 1, gather_idx_c)
        h_fused = torch.cat([h_l1_topk, h_l0_topk], dim=-1)
        h_fused = self.fusion(h_fused)
        h_out = self.decay_attention(h_fused, coords_topk)
        z = h_out.mean(dim=1)
        return self.classifier(z)


print('Model classes defined: DecayAttentionBlock, R2FixedTopKMultiScale.')


## Multi-scale data loader

Cell 9 below replicates v19's `MultiScaleSlideDataset` (cells 0.8b + 0.9) using the slide-key + KDTree coordinate-matching pattern. Per-slide matching indices are cached on first use to `MATCHING_CACHE_DIR`. The dataset returns `(features_l1, features_l0_matched, coords_l1, label, image_id, provider)`.

**Critical alignment detail (verbatim from v19):** L1 is the high-resolution scale (more patches per slide). L0 is the low-resolution scale (fewer patches per slide). For each L1 patch we find the nearest-coordinate L0 patch via scipy's cKDTree, then index `features_l0` by `nn_idx` so the returned `features_l0_matched` is aligned 1:1 with `features_l1` along the patch dimension.

The alignment ASSUMES L0 and L1 coordinates are in the same pixel-coordinate space (slide-level absolute coordinates, not level-relative). The v19 notebook's Cell 0.6 (visual coordinate verification) is reproduced inside Cell 9 below as an automatic sample-of-3 sanity print; if the bounding-box ratios drift > 2x between scales, training is aborted with a clear error.


In [ ]:
# T2.1 Cell 9 - Multi-scale dataset + coordinate matching.
# Verbatim from CLAM_TR_v19_MultiScale_Corrected.ipynb cells 0.8b, 0.9.

def compute_l0_l1_matching(coords_l1: torch.Tensor, coords_l0: torch.Tensor) -> torch.Tensor:
    """For each L1 tile, find the nearest L0 tile via KDTree. Returns long tensor [N_l1]."""
    l1_np = coords_l1.numpy() if isinstance(coords_l1, torch.Tensor) else coords_l1
    l0_np = coords_l0.numpy() if isinstance(coords_l0, torch.Tensor) else coords_l0
    tree = cKDTree(l0_np)
    _, nn_idx = tree.query(l1_np, k=1)
    return torch.tensor(nn_idx, dtype=torch.long)


def _load_pt_features(fpath: str) -> Tuple[torch.Tensor, torch.Tensor]:
    """Load a .pt feature file and return (features [N,D], coords [N,2]).
    Tolerates {features, coords}, {embeddings, coordinates}, or a bare tensor.
    """
    data = torch.load(fpath, map_location='cpu', weights_only=False)
    if isinstance(data, dict):
        feat_key = ('features' if 'features' in data else
                    'embeddings' if 'embeddings' in data else None)
        if feat_key is None:
            raise ValueError(f'No features key in {fpath}; got keys={list(data.keys())}')
        features = data[feat_key]
        if isinstance(features, np.ndarray):
            features = torch.tensor(features, dtype=torch.float32)
        else:
            features = features.float()
        if features.ndim == 3:
            features = features.squeeze(0)
        coord_key = ('coords' if 'coords' in data else
                     'coordinates' if 'coordinates' in data else None)
        if coord_key is None:
            raise ValueError(
                f'No coords/coordinates key in {fpath}; required for multi-scale.\n'
                f'Re-extract features with coord cache enabled.')
        coords = data[coord_key]
        if isinstance(coords, np.ndarray):
            coords = torch.tensor(coords, dtype=torch.float32)
        else:
            coords = coords.float()
        if coords.ndim == 3:
            coords = coords.squeeze(0)
    elif isinstance(data, torch.Tensor):
        raise ValueError(
            f'{fpath} is a bare tensor without coords; multi-scale needs coords.')
    else:
        raise ValueError(f'Unknown .pt format in {fpath}: {type(data)}')
    return features, coords


class MultiScaleSlideDataset(Dataset):
    """v19 cell 0.9, simplified to .pt-only."""

    def __init__(self, df: pd.DataFrame,
                 feature_dir_l1: str, feature_dir_l0: str,
                 cache_dir: str, label_col: str = 'label',
                 max_tiles: Optional[int] = None):
        self.df = df.reset_index(drop=True)
        self.feature_dir_l1 = feature_dir_l1
        self.feature_dir_l0 = feature_dir_l0
        self.cache_dir = cache_dir
        self.label_col = label_col
        self.max_tiles = max_tiles
        os.makedirs(self.cache_dir, exist_ok=True)

    def __len__(self) -> int:
        return len(self.df)

    def _matching_cache_path(self, image_id: str) -> str:
        return os.path.join(self.cache_dir, f'{image_id}_matching.pt')

    def _load_matching(self, image_id: str, coords_l1, coords_l0) -> torch.Tensor:
        cache_path = self._matching_cache_path(image_id)
        if os.path.exists(cache_path):
            return torch.load(cache_path, map_location='cpu', weights_only=False)['nn_idx']
        nn_idx = compute_l0_l1_matching(coords_l1, coords_l0)
        torch.save({'nn_idx': nn_idx}, cache_path)
        return nn_idx

    def __getitem__(self, idx: int):
        row = self.df.iloc[idx]
        image_id = row['image_id']
        label = int(row[self.label_col])
        provider = row.get('data_provider', 'unknown')

        l1_path = os.path.join(self.feature_dir_l1, f'{image_id}.pt')
        l0_path = os.path.join(self.feature_dir_l0, f'{image_id}.pt')
        if not os.path.exists(l1_path):
            raise FileNotFoundError(f'L1 missing for {image_id}: {l1_path}')
        if not os.path.exists(l0_path):
            raise FileNotFoundError(f'L0 missing for {image_id}: {l0_path}')

        features_l1, coords_l1 = _load_pt_features(l1_path)
        features_l0, coords_l0 = _load_pt_features(l0_path)
        nn_idx = self._load_matching(image_id, coords_l1, coords_l0)
        features_l0_matched = features_l0[nn_idx]

        if self.max_tiles is not None and features_l1.shape[0] > self.max_tiles:
            perm = torch.randperm(features_l1.shape[0])[:self.max_tiles]
            features_l1 = features_l1[perm]
            features_l0_matched = features_l0_matched[perm]
            coords_l1 = coords_l1[perm]

        return features_l1, features_l0_matched, coords_l1, label, image_id, provider


def collate_multi_scale(batch):
    features_l1, features_l0, coords_l1, label, image_id, provider = batch[0]
    return (features_l1.unsqueeze(0), features_l0.unsqueeze(0),
            coords_l1.unsqueeze(0), torch.tensor([label]),
            [image_id], [provider])


# === Optional sample-of-3 coordinate sanity check (v19 cell 0.6 lite) ===
def coord_sanity_check(df: pd.DataFrame, k: int = 3):
    """Verify L0 and L1 coordinate bounding boxes overlap on k random slides."""
    sample = df['image_id'].sample(min(k, len(df)), random_state=SEED).tolist()
    print(f'  Coordinate sanity check on {len(sample)} slides:')
    for sid in sample:
        f_l1, c_l1 = _load_pt_features(os.path.join(LEVEL1_FEATURES_DIR, f'{sid}.pt'))
        f_l0, c_l0 = _load_pt_features(os.path.join(LEVEL0_FEATURES_DIR, f'{sid}.pt'))
        bb_l0 = (c_l0[:, 0].min().item(), c_l0[:, 0].max().item())
        bb_l1 = (c_l1[:, 0].min().item(), c_l1[:, 0].max().item())
        span_l0 = bb_l0[1] - bb_l0[0]
        span_l1 = bb_l1[1] - bb_l1[0]
        ratio = span_l1 / (span_l0 + 1e-8) if span_l0 > 0 else float('inf')
        print(f'    {sid[:12]}: L0 N={len(c_l0)} span={span_l0:.0f}, '
              f'L1 N={len(c_l1)} span={span_l1:.0f}, ratio={ratio:.3f}')
        if not (0.5 < ratio < 2.0):
            raise RuntimeError(
                f'L0/L1 coordinate span ratio {ratio:.3f} is outside [0.5, 2.0] for '
                f'slide {sid}. The two scales are likely in DIFFERENT coordinate spaces '
                f'(e.g., one is in level-0 pixels and the other in patch-grid indices). '
                f'Re-check the L1 feature-extraction patch_level argument before training.')
    print('  Coordinate spaces look compatible.')


print('Multi-scale dataset + collate defined.')


## Training loop (one fold)

Cell 11 below packages the v19 training cell (cell 2.1) into a fold-level runner: AdamW + optional linear warmup + CosineAnnealingLR (matches v19 cell 2.1 default), gradient clipping at 1.0, early stopping on validation QWK with patience 10. Per-fold checkpoints save to `CHECKPOINT_DIR/fold{N}_best.pt` and per-fold predictions to `PREDICTIONS_DIR/fold{N}_predictions.json`. Existing checkpoints are detected and the fold is skipped (Colab-disconnect-safe).

**HP_PROFILE flag (set in Cell 5):**
- `'v19_original'` reproduces the §4.8 600-slide config verbatim: `loss=CE_WEIGHTED`, `lr=BEST_LR_FROM_V19`, `K=128`, `dropout=0.15`, `num_heads=4`, `gamma_values=[0.8, 0.9, 0.95, 0.99]`, `epochs=50`, `patience=10`, `weight_decay=1e-5`, cosine annealing, linear warmup 3 epochs.
- `'v20_bestHP'` overlays the v20 best-HP defaults known to work at 9,128: `loss=FOCAL with gamma=1.0`, `lr=2e-4`, `dropout=0.15`, `num_heads=8` (`hidden_dim*2=1024` is divisible by both 4 and 8), `gamma_values=[0.88, 0.90, 0.92, 0.94, 0.96, 0.98, 0.99, 0.995]`, `label_smoothing=0.1` is not applied here because the loss factory doesn't combine focal+smoothing; report this as a known limitation when comparing profiles.

Running BOTH profiles is the most defensible audit because it disentangles (a) the formulation-rejection claim of §4.8 from (b) the hyperparameter-quality-at-scale claim. Mohamed should expect to launch this notebook twice: once with `HP_PROFILE='v19_original'` and once with `'v20_bestHP'`, each producing its own folder under `OUTPUT_DIR`.


In [ ]:
# T2.1 Cell 11 - Training utilities + fold runner.

# === Loss helpers (subset from v19 cell 1.8 + 1.8b) ===
HARD_GRADES_ISUP = [2, 3, 4]

def get_hard_grade_weights(n_classes: int, label_offset: int = 0,
                           hard_boost: float = 2.0) -> torch.Tensor:
    w = torch.ones(n_classes)
    for isup in HARD_GRADES_ISUP:
        lbl = isup - label_offset
        if 0 <= lbl < n_classes:
            w[lbl] = hard_boost
    return w


class FocalLoss(nn.Module):
    """v19 cell 1.8 FocalLoss (gamma=1.0 default matches v18)."""
    def __init__(self, gamma: float = 1.0, weight: torch.Tensor = None):
        super().__init__()
        self.gamma = gamma
        self.weight = weight

    def forward(self, logits, targets):
        ce = F.cross_entropy(logits, targets, weight=self.weight, reduction='none')
        pt = torch.exp(-ce)
        return (((1 - pt) ** self.gamma) * ce).mean()


def create_loss(loss_name: str, n_classes: int, label_offset: int = 0,
                device: torch.device = None):
    """v19 cell 1.8 create_loss subset (the names the §4.8 + alt profiles need)."""
    loss_name = loss_name.upper()
    if loss_name == 'CE':
        return nn.CrossEntropyLoss()
    if loss_name == 'CE_WEIGHTED':
        w = get_hard_grade_weights(n_classes, label_offset).to(device)
        return nn.CrossEntropyLoss(weight=w)
    if loss_name == 'FOCAL':
        return FocalLoss(gamma=1.0)
    if loss_name == 'FOCAL_WEIGHTED':
        w = get_hard_grade_weights(n_classes, label_offset).to(device)
        return FocalLoss(gamma=1.0, weight=w)
    raise ValueError(f'Unknown loss: {loss_name}')


# === HP profiles ===
HP_PROFILES = {
    'v19_original': dict(
        hidden_dim=512, num_heads=4, top_k=128,
        gamma_values=[0.8, 0.9, 0.95, 0.99],
        dropout=0.15, lr=1e-4, weight_decay=1e-5,
        loss_name='CE_WEIGHTED',
        epochs=50, patience=10, warmup_epochs=3, use_cosine=True,
        max_grad_norm=1.0,
    ),
    'v20_bestHP': dict(
        hidden_dim=512, num_heads=8, top_k=128,
        gamma_values=[0.88, 0.90, 0.92, 0.94, 0.96, 0.98, 0.99, 0.995],
        dropout=0.15, lr=2e-4, weight_decay=1e-5,
        loss_name='FOCAL',
        epochs=50, patience=10, warmup_epochs=3, use_cosine=True,
        max_grad_norm=1.0,
    ),
}


# === Metrics (verbatim from v19 cell 0.10) ===
def compute_qwk(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    return float(cohen_kappa_score(y_true, y_pred, weights='quadratic'))

def compute_metrics_full(y_true, y_pred, n_classes, label_offset=0):
    y_t = y_true + label_offset
    y_p = y_pred + label_offset
    qwk = compute_qwk(y_t, y_p)
    acc = float(accuracy_score(y_t, y_p))
    bal_acc = float(balanced_accuracy_score(y_t, y_p))
    per_grade = {}
    for g in range(n_classes + label_offset):
        m = (y_t == g)
        per_grade[int(g)] = float((y_p[m] == g).mean()) if m.any() else 0.0
    hard_acc = float(np.mean([per_grade.get(g, 0.0) for g in HARD_GRADES_ISUP]))
    diff = np.abs(y_t - y_p)
    n = max(1, len(diff))
    off_by_k = {
        'exact':    float((diff == 0).sum() / n * 100),
        'off_1':    float((diff == 1).sum() / n * 100),
        'off_2':    float((diff == 2).sum() / n * 100),
        'off_3plus':float((diff >= 3).sum() / n * 100),
    }
    return {'qwk': qwk, 'accuracy': acc, 'balanced_accuracy': bal_acc,
            'per_grade_recall': per_grade, 'hard_acc': hard_acc,
            'off_by_k': off_by_k}


def get_warmup_scheduler(optimizer, warmup_epochs, warmup_start_lr=1e-7):
    base_lr = optimizer.param_groups[0]['lr']
    warmup_factor = warmup_start_lr / base_lr
    def lr_lambda(epoch):
        if epoch < warmup_epochs:
            return warmup_factor + (1.0 - warmup_factor) * (epoch / warmup_epochs)
        return 1.0
    return LambdaLR(optimizer, lr_lambda)


def train_one_epoch(model, loader, optimizer, criterion, max_grad_norm=1.0):
    model.train()
    total = 0.0
    for batch in loader:
        x_l1, x_l0, coords, labels, _, _ = batch
        x_l1 = x_l1.to(device); x_l0 = x_l0.to(device)
        coords = coords.to(device); labels = labels.to(device)
        logits = model(x_l1, x_l0, coords)
        loss = criterion(logits, labels)
        optimizer.zero_grad()
        loss.backward()
        if max_grad_norm > 0:
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=max_grad_norm)
        optimizer.step()
        total += float(loss.item())
    return total / max(1, len(loader))


@torch.no_grad()
def evaluate(model, loader, n_classes, label_offset, return_preds=False):
    model.eval()
    all_y, all_p, all_ids, all_prov = [], [], [], []
    for batch in loader:
        x_l1, x_l0, coords, labels, image_ids, providers = batch
        x_l1 = x_l1.to(device); x_l0 = x_l0.to(device)
        coords = coords.to(device)
        logits = model(x_l1, x_l0, coords)
        preds = logits.argmax(dim=1).cpu().numpy()
        all_y.extend(labels.numpy()); all_p.extend(preds)
        all_ids.extend(image_ids); all_prov.extend(providers)
    y_true = np.array(all_y); y_pred = np.array(all_p)
    metrics = compute_metrics_full(y_true, y_pred, n_classes, label_offset)
    if return_preds:
        metrics['y_true'] = y_true.tolist()
        metrics['y_pred'] = y_pred.tolist()
        metrics['image_ids'] = list(all_ids)
        metrics['providers'] = list(all_prov)
    return metrics


def fold_runner(fold: int, df_train: pd.DataFrame, df_val: pd.DataFrame,
                n_classes: int, label_offset: int,
                hp: dict, verbose: bool = True):
    """Train v19 multi-scale on a single fold. Returns final metrics + model state."""
    ckpt_path = os.path.join(CHECKPOINT_DIR, f'{HP_PROFILE}_fold{fold}_best.pt')
    pred_path = os.path.join(PREDICTIONS_DIR, f'{HP_PROFILE}_fold{fold}_predictions.json')
    if os.path.exists(ckpt_path) and os.path.exists(pred_path):
        if verbose:
            print(f'  [SKIP] fold {fold} already completed.')
        ckpt = torch.load(ckpt_path, map_location='cpu', weights_only=False)
        with open(pred_path) as f:
            preds = json.load(f)
        return ckpt['metrics'], preds

    train_ds = MultiScaleSlideDataset(df_train, LEVEL1_FEATURES_DIR, LEVEL0_FEATURES_DIR,
                                      MATCHING_CACHE_DIR, label_col='label')
    val_ds = MultiScaleSlideDataset(df_val, LEVEL1_FEATURES_DIR, LEVEL0_FEATURES_DIR,
                                    MATCHING_CACHE_DIR, label_col='label')
    train_loader = DataLoader(train_ds, batch_size=1, shuffle=True,
                              collate_fn=collate_multi_scale, num_workers=0)
    val_loader = DataLoader(val_ds, batch_size=1, shuffle=False,
                            collate_fn=collate_multi_scale, num_workers=0)

    model = R2FixedTopKMultiScale(
        input_dim_l1=1024, input_dim_l0=1024,
        hidden_dim=hp['hidden_dim'], n_classes=n_classes,
        num_heads=hp['num_heads'], gamma=hp['gamma_values'],
        dropout=hp['dropout'], top_k=hp['top_k'],
        learnable_gamma=True).to(device)

    criterion = create_loss(hp['loss_name'], n_classes, label_offset, device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=hp['lr'],
                                  weight_decay=hp['weight_decay'])

    # Scheduler: warmup (if warmup_epochs>0) OR cosine.
    if hp.get('warmup_epochs', 0) > 0:
        scheduler = get_warmup_scheduler(optimizer, hp['warmup_epochs'])
    elif hp.get('use_cosine', True):
        scheduler = CosineAnnealingLR(optimizer, T_max=hp['epochs'])
    else:
        scheduler = None

    best_qwk = -1.0
    best_epoch = -1
    patience_ctr = 0
    best_state = None
    history = []

    pbar = tqdm(range(hp['epochs']), desc=f'fold{fold} ({HP_PROFILE})', leave=True)
    for epoch in pbar:
        train_loss = train_one_epoch(model, train_loader, optimizer, criterion,
                                     hp['max_grad_norm'])
        cur_lr = optimizer.param_groups[0]['lr']
        if scheduler:
            scheduler.step()
        val_metrics = evaluate(model, val_loader, n_classes, label_offset)
        val_qwk = val_metrics['qwk']
        history.append({'epoch': epoch, 'train_loss': train_loss,
                        'val_qwk': val_qwk, 'val_acc': val_metrics['accuracy'],
                        'lr': cur_lr})
        pbar.set_postfix(loss=f'{train_loss:.3f}', QWK=f'{val_qwk:.4f}',
                         lr=f'{cur_lr:.1e}')
        if val_qwk > best_qwk:
            best_qwk = val_qwk
            best_epoch = epoch
            patience_ctr = 0
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        else:
            patience_ctr += 1
        if patience_ctr >= hp['patience']:
            print(f'  Early stop @ epoch {epoch} (best epoch={best_epoch}, QWK={best_qwk:.4f}).')
            break

    if best_state is not None:
        model.load_state_dict(best_state)
    model.to(device)
    final = evaluate(model, val_loader, n_classes, label_offset, return_preds=True)

    torch.save({
        'fold': fold, 'hp_profile': HP_PROFILE,
        'model_state_dict': best_state,
        'metrics': {k: v for k, v in final.items()
                    if k not in {'y_true', 'y_pred', 'image_ids', 'providers'}},
        'best_epoch': best_epoch, 'best_qwk': best_qwk,
        'history': history, 'hp': hp,
    }, ckpt_path)

    with open(pred_path, 'w') as f:
        json.dump({'fold': fold, 'hp_profile': HP_PROFILE,
                   'y_true': final['y_true'], 'y_pred': final['y_pred'],
                   'image_ids': final['image_ids'], 'providers': final['providers'],
                   'qwk': final['qwk']}, f)

    print(f'  fold{fold} DONE: QWK={final["qwk"]:.4f}, Acc={final["accuracy"]:.4f}, '
          f'best_epoch={best_epoch}, ckpt={ckpt_path}')
    return final, {'y_true': final['y_true'], 'y_pred': final['y_pred'],
                   'image_ids': final['image_ids'], 'providers': final['providers']}


print('Training utilities + fold_runner defined.')


## Main sweep: 5 folds x 1 variant

Cell 13 below loads the 9,128 noise-cleaned CSV, applies the v19 5-class convention (drop ISUP=0, map grades 1-5 to labels 0-4 with label_offset=1, matching v19 cell 0.7), filters to the L0 INTERSECTION L1 intersection, and runs `fold_runner` for each of the 5 folds. When `TEST_MODE=True` only fold 0 runs (sanity check; ~3-4 hr at A100). When `TEST_MODE=False` all 5 folds run sequentially (~12-20 hr total).

**Note on label space:** The §4.8 MS_TopK experiment is a 5-class run (grades 1-5; ISUP 0 is excluded). The v20 fullscale factorial that the comparison target `RetNet_bestHP=0.9543` reports is a 6-class run. The QWK numbers are NOT directly comparable on the same scale; this notebook reproduces the §4.8 5-class convention so that the comparison to the §4.8 600-slide MS_TopK QWK 0.6262 is apples-to-apples. The §4.6 RetNet_bestHP 0.9543 should be re-computed in the 5-class subset (Mohamed has the v20 fold predictions; a quick re-evaluation script can convert them to 5-class QWK for fair comparison; this is OUT OF SCOPE for this notebook but flagged in Cell 17).


In [ ]:
# T2.1 Cell 13 - Main sweep.

# Load CSV and apply v19 5-class convention.
df_full = pd.read_csv(SPLIT_CSV_PATH)
print(f'CSV: {len(df_full):,} rows.')

# Convert kfold 1-5 -> 0-4 if needed (matches v20 fullscale cell 37).
kmin, kmax = df_full['kfold'].min(), df_full['kfold'].max()
print(f'kfold range: {kmin}..{kmax}')
if kmin == 1:
    print('  Converting kfold 1-5 -> 0-4.')
    df_full['kfold'] = df_full['kfold'] - 1
assert df_full['kfold'].min() == 0 and df_full['kfold'].max() == 4, \
    f'kfold must be 0..4 after conversion; got {df_full["kfold"].min()}..{df_full["kfold"].max()}'
df_full = df_full.rename(columns={'kfold': 'fold'})

# Filter to slides with BOTH L0 and L1 features.
import glob
l0_ids = {os.path.splitext(os.path.basename(p))[0]
          for p in glob.glob(os.path.join(LEVEL0_FEATURES_DIR, f'*{L0_EXT}'))}
l1_ids = {os.path.splitext(os.path.basename(p))[0]
          for p in glob.glob(os.path.join(LEVEL1_FEATURES_DIR, f'*{L1_EXT}'))}
both_ids = l0_ids & l1_ids
df = df_full[df_full['image_id'].astype(str).isin(both_ids)].copy().reset_index(drop=True)
print(f'  After L0 INTERSECTION L1 filter: {len(df):,}/{len(df_full):,} slides.')

# v19 5-class convention: drop ISUP=0, remap 1-5 -> 0-4.
df_5c = df[df['isup_grade'] > 0].copy().reset_index(drop=True)
df_5c['label'] = df_5c['isup_grade'] - 1
print(f'  5-class subset (grades 1-5): {len(df_5c):,} slides.')
print(f'  Grade distribution:\n{df_5c["isup_grade"].value_counts().sort_index()}')

# Coordinate sanity check (3 random slides).
try:
    coord_sanity_check(df_5c, k=3)
except Exception as e:
    print(f'  Coordinate sanity check failed: {e}')
    raise

# Pick HP profile.
hp = HP_PROFILES[HP_PROFILE]
print(f'\nHP_PROFILE={HP_PROFILE}:')
for k, v in hp.items():
    print(f'  {k}: {v}')

# Sweep.
n_classes = 5
label_offset = 1
folds_to_run = [0] if TEST_MODE else [0, 1, 2, 3, 4]
print(f'\nRunning folds: {folds_to_run}  (TEST_MODE={TEST_MODE})')

ALL_FOLD_METRICS = []
for fold in folds_to_run:
    print(f'\n========== FOLD {fold} ==========')
    train_df = df_5c[df_5c['fold'] != fold].reset_index(drop=True)
    val_df = df_5c[df_5c['fold'] == fold].reset_index(drop=True)
    print(f'  train={len(train_df):,}, val={len(val_df):,}')
    final_metrics, preds = fold_runner(fold, train_df, val_df, n_classes, label_offset,
                                       hp, verbose=True)
    ALL_FOLD_METRICS.append({'fold': fold, **{k: v for k, v in final_metrics.items()
                                              if k not in {'y_true', 'y_pred', 'image_ids', 'providers'}}})

# Persist combined summary.
summary_path = os.path.join(OUTPUT_DIR, f'{HP_PROFILE}_summary.json')
with open(summary_path, 'w') as f:
    json.dump({
        'hp_profile': HP_PROFILE,
        'n_folds_run': len(folds_to_run),
        'folds_run': folds_to_run,
        'test_mode': TEST_MODE,
        'fold_metrics': ALL_FOLD_METRICS,
        'qwk_per_fold': [m['qwk'] for m in ALL_FOLD_METRICS],
        'qwk_mean': float(np.mean([m['qwk'] for m in ALL_FOLD_METRICS])) if ALL_FOLD_METRICS else float('nan'),
        'qwk_std':  float(np.std([m['qwk'] for m in ALL_FOLD_METRICS], ddof=1)) if len(ALL_FOLD_METRICS) > 1 else float('nan'),
    }, f, indent=2)
print(f'\nSummary saved to {summary_path}')


## Compare against §4.8 600-slide rejection

Cell 15 below loads the 5 fold QWKs from Cell 13 and computes a paired-difference analysis against the 600-slide §4.8 number. Because the 5 folds at 9,128 scale are independent runs (different model instances; different training subsets) from the 5 folds at 600 scale, a paired t-test on per-fold deltas is the same statistical framework §4.6 uses for the RetNet-vs-ABMIL 600-vs-9,128 comparison.

The 600-slide MS_TopK QWK 0.6262 +/- 0.0798 is reproduced verbatim from `CLAM-master/CLAM_TR_Results/v19_multiscale/v19_final_summary.json` (the file already committed to the repo on 2026-01-27). The five 600-slide fold QWKs needed for the paired test are extracted from that file's `fold_qwks` field (filled in below).


In [ ]:
# T2.1 Cell 15 - Paired comparison vs §4.8 600-slide MS_TopK.

# §4.8 anchor (verified verbatim from v19_final_summary.json on 2026-01-27).
V19_600_SLIDE_QWK_MEAN = 0.6262
V19_600_SLIDE_QWK_STD  = 0.0798
# Per-fold values needed for paired t-test. The published summary stores qwk_mean / qwk_std only;
# the actual per-fold QWKs live in the v19 checkpoint files. Mohamed must fill these in by reading
# CLAM-master/CLAM_TR_Results/v19_multiscale/checkpoints/MS_TopK/fold{1..5}_best.pt and copying
# the 'qwk' field per fold. If unavailable, the script falls back to a one-sample t-test against
# the 600-slide mean (less powerful but still valid).
V19_600_SLIDE_PER_FOLD = None
# Example (commented out -- replace with the actual numbers and uncomment):
# V19_600_SLIDE_PER_FOLD = [0.55, 0.62, 0.68, 0.64, 0.64]  # 5 folds, MS_TopK @ 600 slides

# Load 9,128 results from Cell 13.
if not ALL_FOLD_METRICS:
    raise RuntimeError('No fold metrics found. Run Cell 13 first.')
qwks_9128 = np.array([m['qwk'] for m in ALL_FOLD_METRICS], dtype=float)
print(f'9,128 v19 multi-scale QWK per fold: {qwks_9128.tolist()}')
print(f'  mean = {qwks_9128.mean():.4f}, '
      f'std = {qwks_9128.std(ddof=1) if len(qwks_9128) > 1 else float("nan"):.4f}')
print(f'§4.8 600-slide v19 multi-scale (MS_TopK): '
      f'QWK = {V19_600_SLIDE_QWK_MEAN:.4f} +/- {V19_600_SLIDE_QWK_STD:.4f}.')

if V19_600_SLIDE_PER_FOLD is not None and len(V19_600_SLIDE_PER_FOLD) == len(qwks_9128):
    delta = qwks_9128 - np.array(V19_600_SLIDE_PER_FOLD, dtype=float)
    t_stat, p_value = stats.ttest_rel(qwks_9128, V19_600_SLIDE_PER_FOLD)
    se = stats.sem(delta)
    ci = stats.t.interval(0.95, df=len(delta) - 1, loc=delta.mean(), scale=se)
    print('\nPaired t-test (9,128 - 600), 5 folds:')
    print(f'  mean delta = {delta.mean():.4f}, std = {delta.std(ddof=1):.4f}')
    print(f'  95% CI on delta = [{ci[0]:.4f}, {ci[1]:.4f}]')
    print(f'  t = {t_stat:.3f}, p = {p_value:.4g}')
    significant = bool(p_value < 0.05)
elif len(qwks_9128) > 1:
    t_stat, p_value = stats.ttest_1samp(qwks_9128, V19_600_SLIDE_QWK_MEAN)
    print('\nOne-sample t-test (9,128 mean vs §4.8 mean), 5 folds:')
    print(f'  9,128 mean - 600 mean = {qwks_9128.mean() - V19_600_SLIDE_QWK_MEAN:.4f}')
    print(f'  t = {t_stat:.3f}, p = {p_value:.4g}')
    significant = bool(p_value < 0.05)
else:
    significant = None
    print('\nOnly one fold available (TEST_MODE); skipping statistical test.')

if significant is not None:
    direction = 'higher' if qwks_9128.mean() > V19_600_SLIDE_QWK_MEAN else 'lower'
    print(f'\n9,128 scale QWK is {direction} than 600 scale.'
          f' Difference is {"significant" if significant else "NOT significant"} at alpha=0.05.')


## Output for thesis

Cell 17 below prints a defence-ready summary sentence Mohamed can paste into Section 4.8 / Section 4.6 of the thesis to close T2.1. The sentence states the 9,128-scale v19 multi-scale QWK, compares it to (a) the §4.8 600-slide MS_TopK rejection number and (b) the §3 RetNet_bestHP 9,128-scale anchor (caveat: cross-class re-scoring needed for fair comparison; see Cell 13 note).


In [ ]:
# T2.1 Cell 17 - Defence-ready summary.

# v20 RetNet_bestHP anchor (verbatim from §3 / v20_final_summary.json; 6-class scale).
# WARNING: this is a 6-class QWK; the 5-class v19 number computed above is on the
# different label space (grades 1-5). For an apples-to-apples comparison you must
# either (a) re-evaluate v20 RetNet_bestHP fold predictions in 5-class space, or
# (b) re-evaluate the v19 multi-scale fold predictions in 6-class space (the
# original v19 model has a 5-class head and would need retraining; option (a) is
# far cheaper -- it's a one-shot recomputation using the saved per-fold predictions
# under CLAM-master/CLAM_TR_Results/v20_fullscale/UNI/predictions/).
RETNET_BESTHP_QWK_6CLASS_MEAN = 0.9543
RETNET_BESTHP_QWK_6CLASS_STD  = 0.0046

qwks_9128 = np.array([m['qwk'] for m in ALL_FOLD_METRICS], dtype=float)
mean_9128 = float(qwks_9128.mean())
std_9128  = float(qwks_9128.std(ddof=1)) if len(qwks_9128) > 1 else float('nan')

delta_vs_600 = mean_9128 - V19_600_SLIDE_QWK_MEAN
delta_vs_retnet = mean_9128 - RETNET_BESTHP_QWK_6CLASS_MEAN

# Within-noise band: 1 sigma of v19 600-slide std + 1 sigma of 9,128 std.
noise_band = V19_600_SLIDE_QWK_STD + std_9128 if not np.isnan(std_9128) else V19_600_SLIDE_QWK_STD
if abs(delta_vs_600) > 2 * noise_band:
    verdict_vs_600 = 'CONFIRMED at scale (9,128 still below the §3 anchor)' if delta_vs_600 < 0 \
                     else 'OVERTURNED at scale (9,128 substantially exceeds the §4.8 number)'
elif abs(delta_vs_600) > noise_band:
    verdict_vs_600 = 'INCONCLUSIVE at scale (>1 noise band, <2 noise bands)'
else:
    verdict_vs_600 = 'WITHIN NOISE (no meaningful change of QWK with scale)'

print('=' * 78)
print('T2.1 DEFENCE-READY SUMMARY')
print('=' * 78)
print(f"v19 multi-scale ({HP_PROFILE}) at 9,128 slides reached "
      f"QWK = {mean_9128:.4f} +/- {std_9128:.4f} (5-class, grades 1-5; "
      f"5 folds, label_offset=1, mean +/- 1 SD).")
print(f"vs §4.8 600-slide MS_TopK {V19_600_SLIDE_QWK_MEAN:.4f} +/- {V19_600_SLIDE_QWK_STD:.4f}: "
      f"delta = {delta_vs_600:+.4f} -> {verdict_vs_600}.")
print(f"vs §3 RetNet_bestHP {RETNET_BESTHP_QWK_6CLASS_MEAN:.4f} +/- {RETNET_BESTHP_QWK_6CLASS_STD:.4f}: "
      f"delta = {delta_vs_retnet:+.4f}.")
print()
print('IMPORTANT CAVEAT: the v19 multi-scale 5-class QWK above is NOT directly')
print('comparable to the §3 RetNet_bestHP 6-class QWK because the label spaces')
print('differ (grades 1-5 vs 0-5). Apples-to-apples comparison requires re-evaluating')
print('the v20 RetNet_bestHP fold predictions in 5-class space using sklearn:')
print("  from sklearn.metrics import cohen_kappa_score")
print("  y_true_5c = [y for y in y_true_6c if y > 0]")
print("  y_pred_5c = [p for y, p in zip(y_true_6c, y_pred_6c) if y > 0]")
print("  qwk_5c = cohen_kappa_score(y_true_5c, y_pred_5c, weights='quadratic')")
print('See Section 4.8 reconciliation discussion for the full statistical framework.')
print('=' * 78)
